# 3D screened Keller–Segel: dataset generation

Data-only extraction from `GRNO_KellerSegel3D_64.ipynb`. Numerical solver implementations, physical parameters, split seeds, and original profile defaults are preserved. Neural models, training, checkpoint evaluation, and model-result plotting are omitted. Source cell numbers are recorded in each code cell's metadata.

The saved default is `PROFILE = "smoke"`. Read the profile table in [README.md](README.md) before running: the `paper` profile can require substantial GPU time and disk space. Install a compatible PyTorch build plus NumPy, pandas, tqdm, and IPython in this notebook's kernel. These notebooks do not install packages automatically.

Run from the repository, or set the environment variable `GRNO_ROOT` to your intended project/data location before starting the kernel. The printed `ROOT` and data roots identify the generated files. Restart and run from the top after changing profiles. Set `RUN_DATA_GENERATION = False` to inspect a complete existing cache; simulator audits and normalization still run.

The `smoke` profile stores 16³ arrays and is a code-path check. Select `paper` for 64³ data and the Table 6 split sizes. The source counts `*_transitions`; the stored shape includes the initial frame. Both original simulator audits and the original training-only statistics estimator are retained. Legacy `RunConfig` values are retained but no training or window-manifest generation occurs.


In [ ]:
from __future__ import annotations

import contextlib
import dataclasses
import hashlib
import json
import math
import os
import random
import shutil
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from IPython.display import display
from tqdm.auto import tqdm

# Storage location only: set GRNO_ROOT before starting this kernel to override it.
# No solver, split, seed, or physical parameter is changed by this root selection.
def locate_project_root() -> Path:
    override = os.environ.get("GRNO_ROOT")
    if override:
        return Path(override).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "README.md").is_file() and (candidate / "notebooks").is_dir() and (candidate / "data").is_dir():
            return candidate
    return cwd

ROOT = locate_project_root()
ROOT.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print({"project_root": str(ROOT), "device": str(device), "torch": torch.__version__})


In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def short_hash(payload) -> str:
    raw = json.dumps(payload, sort_keys=True, default=str).encode()
    return hashlib.sha256(raw).hexdigest()[:12]


def atomic_json(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".partial")
    tmp.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str))
    os.replace(tmp, path)


## Original profiles and PDE parameters


In [ ]:
PROFILE = "smoke"                 # "smoke", "pilot", or "paper"


RUN_DATA_GENERATION = True


RUN_SPATIAL_REFINEMENT = PROFILE in ("smoke", "paper")


DATASET_VERSION = "screened-keller-segel3d-etd2-v1"


@dataclass(frozen=True)
class RunConfig:
    profile: str
    epochs: int
    samples_per_epoch: int
    micro_batch: int
    grad_accum: int
    train_rollout: int
    val_rollout: int
    test_rollout: int
    width: int
    depths: tuple
    lr: float = 3e-4
    weight_decay: float = 1e-4
    grad_weight: float = 0.05
    spectral_weight: float = 0.02
    rollout_decay: float = 0.8
    use_amp: bool = True

    @property
    def optimizer_updates_per_epoch(self):
        denominator = self.micro_batch * self.grad_accum
        if self.samples_per_epoch % denominator:
            raise ValueError("samples_per_epoch must be divisible by effective batch size")
        return self.samples_per_epoch // denominator

    @property
    def effective_batch(self):
        return self.micro_batch * self.grad_accum


@dataclass(frozen=True)
class KellerConfig:
    n: int
    length: float
    diffusivity: float
    chemotaxis: float
    screening: float
    frame_dt: float
    substeps_per_frame: int
    mean_density: float
    initial_log_amplitude: float
    initial_max_mode: int
    n_train: int
    n_val: int
    n_test: int
    train_transitions: int
    val_transitions: int
    test_transitions: int


if PROFILE == "smoke":
    RUN = RunConfig(PROFILE, epochs=2, samples_per_epoch=8, micro_batch=1,
                    grad_accum=2, train_rollout=2, val_rollout=3,
                    test_rollout=5, width=8, depths=(1, 1, 1, 1),
                    lr=1e-3, spectral_weight=0.0, use_amp=False)
    KSCFG = KellerConfig(
        n=16, length=2 * math.pi, diffusivity=0.10, chemotaxis=0.30,
        screening=1.0, frame_dt=0.25, substeps_per_frame=4,
        mean_density=1.0, initial_log_amplitude=0.20, initial_max_mode=3,
        n_train=2, n_val=1, n_test=1,
        train_transitions=6, val_transitions=8, test_transitions=10)
elif PROFILE == "pilot":
    RUN = RunConfig(PROFILE, epochs=20, samples_per_epoch=128, micro_batch=1,
                    grad_accum=4, train_rollout=4, val_rollout=10,
                    test_rollout=40, width=16, depths=(1, 1, 2, 2))
    KSCFG = KellerConfig(
        n=64, length=2 * math.pi, diffusivity=0.10, chemotaxis=0.30,
        screening=1.0, frame_dt=0.25, substeps_per_frame=8,
        mean_density=1.0, initial_log_amplitude=0.20, initial_max_mode=3,
        n_train=8, n_val=2, n_test=2,
        train_transitions=50, val_transitions=100, test_transitions=100)
elif PROFILE == "paper":
    RUN = RunConfig(PROFILE, epochs=150, samples_per_epoch=512, micro_batch=1,
                    grad_accum=4, train_rollout=4, val_rollout=10,
                    test_rollout=40, width=24, depths=(2, 2, 3, 4))
    KSCFG = KellerConfig(
        n=64, length=2 * math.pi, diffusivity=0.10, chemotaxis=0.30,
        screening=1.0, frame_dt=0.25, substeps_per_frame=8,
        mean_density=1.0, initial_log_amplitude=0.20, initial_max_mode=3,
        n_train=64, n_val=8, n_test=12,
        train_transitions=50, val_transitions=100, test_transitions=100)
else:
    raise ValueError(f"Unknown PROFILE={PROFILE!r}")


if PROFILE != "smoke" and device.type != "cuda":
    raise RuntimeError("pilot/paper profiles require a CUDA GPU; use smoke for CPU checks")


DATA_ROOT = ROOT / "data" / "keller_segel3d" / PROFILE


DATA_ROOT.mkdir(parents=True, exist_ok=True)
print(json.dumps({"profile": PROFILE, "pde": asdict(KSCFG), "data_root": str(DATA_ROOT)}, indent=2))


## Original Fourier ETD2 reference solver


In [ ]:
class KellerSegelETD2:
    """Dealiased Fourier ETD2 reference integrator, not part of GRNO inference."""

    def __init__(self, cfg, target_device=device):
        self.cfg = cfg
        self.device = target_device
        n, L = cfg.n, cfg.length
        k1 = 2 * math.pi * torch.fft.fftfreq(n, d=L / n, device=target_device)
        kz, ky, kx = torch.meshgrid(k1, k1, k1, indexing="ij")
        k2 = kx.square() + ky.square() + kz.square()
        cutoff = n // 3
        indices = torch.fft.fftfreq(n, device=target_device) * n
        iz, iy, ix = torch.meshgrid(indices, indices, indices, indexing="ij")
        dealias = ((ix.abs() <= cutoff) & (iy.abs() <= cutoff) &
                   (iz.abs() <= cutoff)).float()
        dealias[0, 0, 0] = 1.0
        green = torch.where(k2 > 0, 1.0 / (k2 + cfg.screening ** 2),
                            torch.zeros_like(k2))
        self.kx, self.ky, self.kz = kx, ky, kz
        self.k2, self.dealias, self.green = k2, dealias, green
        h = cfg.frame_dt / cfg.substeps_per_frame
        z = -cfg.diffusivity * k2 * h
        expz = torch.exp(z)
        small = z.abs() < 1e-6
        phi1 = torch.where(small, 1 + z / 2 + z.square() / 6,
                           torch.expm1(z) / z)
        phi2 = torch.where(small, 0.5 + z / 6 + z.square() / 24,
                           (torch.expm1(z) - z) / z.square())
        self.h, self.expz, self.phi1, self.phi2 = h, expz, phi1, phi2

    def physical(self, density_hat):
        return torch.fft.ifftn(density_hat, dim=(-3, -2, -1)).real

    def density_hat(self, density):
        return torch.fft.fftn(density.float(), dim=(-3, -2, -1)) * self.dealias

    def carrier_from_hat(self, density_hat):
        potential_hat = density_hat * self.green
        ux = torch.fft.ifftn(1j * self.kx * potential_hat,
                             dim=(-3, -2, -1)).real
        uy = torch.fft.ifftn(1j * self.ky * potential_hat,
                             dim=(-3, -2, -1)).real
        uz = torch.fft.ifftn(1j * self.kz * potential_hat,
                             dim=(-3, -2, -1)).real
        return self.cfg.chemotaxis * torch.stack((ux, uy, uz), dim=1)

    def nonlinear_hat(self, density_hat):
        density = self.physical(density_hat)
        velocity = self.carrier_from_hat(density_hat)
        flux_x = density * velocity[:, 0]
        flux_y = density * velocity[:, 1]
        flux_z = density * velocity[:, 2]
        divergence_hat = 1j * (
            self.kx * torch.fft.fftn(flux_x, dim=(-3, -2, -1))
            + self.ky * torch.fft.fftn(flux_y, dim=(-3, -2, -1))
            + self.kz * torch.fft.fftn(flux_z, dim=(-3, -2, -1))
        )
        return -divergence_hat * self.dealias

    def substep(self, density_hat):
        n0 = self.nonlinear_hat(density_hat)
        stage = self.expz * density_hat + self.h * self.phi1 * n0
        stage = stage * self.dealias
        n1 = self.nonlinear_hat(stage)
        result = stage + self.h * self.phi2 * (n1 - n0)
        return result * self.dealias

    def frame(self, density_hat):
        for _ in range(self.cfg.substeps_per_frame):
            density_hat = self.substep(density_hat)
        return density_hat

    def random_initial_hat(self, seed):
        generator = torch.Generator(device=self.device).manual_seed(int(seed))
        noise = torch.randn(1, self.cfg.n, self.cfg.n, self.cfg.n,
                            generator=generator, device=self.device)
        noise_hat = torch.fft.fftn(noise, dim=(-3, -2, -1))
        k_radius2 = self.k2
        low_pass = ((k_radius2 >= 1) &
                    (k_radius2 <= self.cfg.initial_max_mode ** 2)).float()
        smooth = self.physical(noise_hat * low_pass)
        smooth = smooth / smooth.std().clamp_min(1e-7)
        density = torch.exp(self.cfg.initial_log_amplitude * smooth)
        density = density * (self.cfg.mean_density / density.mean())
        return self.density_hat(density)

    def analytic_initial_hat(self):
        n, L = self.cfg.n, self.cfg.length
        z, y, x = torch.meshgrid(
            torch.arange(n, device=self.device) * L / n,
            torch.arange(n, device=self.device) * L / n,
            torch.arange(n, device=self.device) * L / n,
            indexing="ij")
        field = (0.9 * torch.cos(x + 0.2) + 0.6 * torch.sin(2 * y - 0.3)
                 + 0.5 * torch.cos(z + y) + 0.35 * torch.sin(x - z))
        field = field / field.std().clamp_min(1e-7)
        density = torch.exp(self.cfg.initial_log_amplitude * field)
        density = density * (self.cfg.mean_density / density.mean())
        return self.density_hat(density[None])

    def free_energy(self, density_hat):
        density = self.physical(density_hat)
        fluctuation_hat = density_hat.clone()
        fluctuation_hat[..., 0, 0, 0] = 0
        potential = self.physical(fluctuation_hat * self.green)
        entropy = (density.clamp_min(1e-8) *
                   density.clamp_min(1e-8).log()).mean((-3, -2, -1))
        attraction = ((density - density.mean((-3, -2, -1), keepdim=True))
                      * potential).mean((-3, -2, -1))
        return self.cfg.diffusivity * entropy - 0.5 * self.cfg.chemotaxis * attraction

    def diagnostics(self, density_hat):
        density = self.physical(density_hat)
        velocity = self.carrier_from_hat(density_hat)
        mass = density.mean((-3, -2, -1))
        speed = velocity.square().sum(1).sqrt()
        dx = self.cfg.length / self.cfg.n
        cfl = speed.amax((-3, -2, -1)) * self.h / dx
        fluctuation_hat = density_hat.clone()
        fluctuation_hat[..., 0, 0, 0] = 0
        outer = (self.k2 > (self.cfg.n / 4) ** 2)
        energy = fluctuation_hat.abs().square()
        outer_fraction = (energy[..., outer].sum(-1) /
                          energy.flatten(1).sum(-1).clamp_min(1e-30))
        negative_mass = (-density).clamp_min(0).sum((-3, -2, -1))
        positive_mass = density.clamp_min(0).sum((-3, -2, -1)).clamp_min(1e-12)
        return {
            "mass": float(mass.mean()),
            "minimum_density": float(density.amin()),
            "maximum_density": float(density.amax()),
            "density_std": float(density.std()),
            "negative_mass_fraction": float((negative_mass / positive_mass).mean()),
            "carrier_rms": float(velocity.square().mean().sqrt()),
            "internal_cfl_max": float(cfl.max()),
            "outer_band_energy_fraction": float(outer_fraction.mean()),
            "free_energy": float(self.free_energy(density_hat).mean()),
        }


SIM = KellerSegelETD2(KSCFG)
print("Reference simulator ready. It is used only to create labels/audits, never by GRNO.")


## Required original simulator and spatial-refinement audits


In [ ]:
@torch.no_grad()
def simulator_preflight():
    initial_hat = SIM.random_initial_hat(918273)
    initial_mass = SIM.diagnostics(initial_hat)["mass"]
    state_hat = initial_hat
    energies = [SIM.diagnostics(state_hat)["free_energy"]]
    for _ in range(min(10, KSCFG.test_transitions)):
        state_hat = SIM.frame(state_hat)
        energies.append(SIM.diagnostics(state_hat)["free_energy"])
    diagnostics = SIM.diagnostics(state_hat)
    diagnostics["mass_relative_drift"] = abs(
        diagnostics["mass"] - initial_mass) / max(abs(initial_mass), 1e-12)
    diagnostics["max_free_energy_increase"] = float(
        np.maximum(np.diff(energies), 0).max(initial=0.0))

    refined_cfg = dataclasses.replace(
        KSCFG, substeps_per_frame=2 * KSCFG.substeps_per_frame)
    refined = KellerSegelETD2(refined_cfg)
    coarse_one = SIM.frame(state_hat)
    refined_one = refined.frame(state_hat)
    temporal_error = float(
        torch.linalg.vector_norm(SIM.physical(coarse_one - refined_one))
        / torch.linalg.vector_norm(refined.physical(refined_one)).clamp_min(1e-12))
    diagnostics["one_frame_temporal_refinement_relative_l2"] = temporal_error
    print(json.dumps(diagnostics, indent=2))

    assert math.isfinite(temporal_error) and temporal_error < 1e-2
    assert diagnostics["mass_relative_drift"] < 1e-5
    assert diagnostics["minimum_density"] > -5e-4
    assert diagnostics["internal_cfl_max"] < 0.8
    assert diagnostics["outer_band_energy_fraction"] < 2e-2
    if temporal_error > 3e-3:
        warnings.warn("Temporal refinement error is above the preferred 0.3% paper gate")
    if diagnostics["max_free_energy_increase"] > 2e-5:
        warnings.warn("Discrete free energy was not monotone to the diagnostic tolerance")
    return diagnostics


@torch.no_grad()
def spatial_refinement_preflight():
    if not RUN_SPATIAL_REFINEMENT:
        print("Spatial refinement skipped. Set RUN_SPATIAL_REFINEMENT=True before a paper run.")
        return None
    coarse = KellerSegelETD2(KSCFG)
    fine_cfg = dataclasses.replace(KSCFG, n=2 * KSCFG.n)
    fine = KellerSegelETD2(fine_cfg)
    coarse_hat, fine_hat = coarse.analytic_initial_hat(), fine.analytic_initial_hat()
    for _ in range(2):
        coarse_hat, fine_hat = coarse.frame(coarse_hat), fine.frame(fine_hat)
    coarse_density = coarse.physical(coarse_hat)
    fine_on_coarse = fine.physical(fine_hat)[..., ::2, ::2, ::2]
    error = float(torch.linalg.vector_norm(coarse_density - fine_on_coarse)
                  / torch.linalg.vector_norm(fine_on_coarse).clamp_min(1e-12))
    result = {"coarse_n": KSCFG.n, "fine_n": fine_cfg.n,
              "two_frame_spatial_refinement_relative_l2": error}
    print(result)
    assert math.isfinite(error) and error < 3e-2
    if error > 1e-2:
        warnings.warn("Spatial refinement error is above the preferred 1% paper gate")
    del fine, fine_hat, fine_on_coarse
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


SIMULATOR_PREFLIGHT = simulator_preflight()
SPATIAL_PREFLIGHT = spatial_refinement_preflight()


## Generate/resume per-trajectory files


In [ ]:
SPLIT_COUNTS = {"train": KSCFG.n_train, "val": KSCFG.n_val, "test": KSCFG.n_test}
SPLIT_TRANSITIONS = {
    "train": KSCFG.train_transitions,
    "val": KSCFG.val_transitions,
    "test": KSCFG.test_transitions,
}
SPLIT_SEED_OFFSETS = {"train": 100_000, "val": 200_000, "test": 300_000}
DATA_SIGNATURE = short_hash({"version": DATASET_VERSION, "pde": asdict(KSCFG)})
METADATA_PATH = DATA_ROOT / "metadata.json"


def trajectory_path(split, trajectory):
    return DATA_ROOT / split / f"trajectory_{trajectory:04d}.npy"


def expected_shape(split):
    return (SPLIT_TRANSITIONS[split] + 1, 1, KSCFG.n, KSCFG.n, KSCFG.n)


def validate_existing_trajectory(path, split):
    try:
        array = np.load(path, mmap_mode="r")
        valid = array.shape == expected_shape(split) and array.dtype == np.float32
        del array
        return valid
    except Exception:
        return False


@torch.no_grad()
def final_file_diagnostics(path):
    array = np.load(path, mmap_mode="r")
    first = torch.from_numpy(np.asarray(array[0]).copy()).to(device)
    final = torch.from_numpy(np.asarray(array[-1]).copy()).to(device)
    first_hat, final_hat = SIM.density_hat(first), SIM.density_hat(final)
    result = SIM.diagnostics(final_hat)
    initial_mass = SIM.diagnostics(first_hat)["mass"]
    result["mass_relative_drift"] = abs(result["mass"] - initial_mass) / max(
        abs(initial_mass), 1e-12)
    return result


@torch.no_grad()
def generate_trajectory(split, trajectory):
    path = trajectory_path(split, trajectory)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and validate_existing_trajectory(path, split):
        return {"split": split, "trajectory": trajectory, "status": "reused",
                **final_file_diagnostics(path)}
    if path.exists():
        raise RuntimeError(f"Existing trajectory has the wrong schema: {path}")

    temporary = path.with_suffix(".partial.npy")
    if temporary.exists():
        temporary.unlink()
    output = np.lib.format.open_memmap(
        temporary, mode="w+", dtype=np.float32, shape=expected_shape(split))
    state_hat = SIM.random_initial_hat(SPLIT_SEED_OFFSETS[split] + trajectory)
    initial_diag = SIM.diagnostics(state_hat)
    output[0] = SIM.physical(state_hat).cpu().numpy().astype(np.float32)
    progress = tqdm(range(SPLIT_TRANSITIONS[split]),
                    desc=f"generate Keller3D {split} {trajectory}", leave=False)
    for frame in progress:
        state_hat = SIM.frame(state_hat)
        density = SIM.physical(state_hat)
        if not torch.isfinite(density).all():
            raise FloatingPointError(f"Non-finite density in {split}/{trajectory}, frame {frame + 1}")
        output[frame + 1] = density.cpu().numpy().astype(np.float32)
        if (frame + 1) % 10 == 0 or frame + 1 == SPLIT_TRANSITIONS[split]:
            diag = SIM.diagnostics(state_hat)
            progress.set_postfix(min=f"{diag['minimum_density']:.3f}",
                                 max=f"{diag['maximum_density']:.3f}",
                                 cfl=f"{diag['internal_cfl_max']:.2f}")
            if diag["minimum_density"] < -5e-3 or diag["maximum_density"] > 50:
                raise RuntimeError(
                    "Reference trajectory left the validated stable regime: " + str(diag))
            if diag["internal_cfl_max"] >= 0.8:
                raise RuntimeError("Reference solver CFL gate failed: " + str(diag))
    output.flush()
    del output
    os.replace(temporary, path)
    final_diag = SIM.diagnostics(state_hat)
    final_diag["mass_relative_drift"] = abs(
        final_diag["mass"] - initial_diag["mass"]) / max(abs(initial_diag["mass"]), 1e-12)
    if final_diag["mass_relative_drift"] >= 1e-5:
        raise RuntimeError("Generated trajectory violates mass conservation")
    return {"split": split, "trajectory": trajectory, "status": "generated",
            **final_diag}


required_bytes = sum(
    SPLIT_COUNTS[split] * np.prod(expected_shape(split)) * np.dtype(np.float32).itemsize
    for split in SPLIT_COUNTS)
free_bytes = shutil.disk_usage(DATA_ROOT).free
print({"estimated_complete_dataset_GB": round(required_bytes / 1e9, 3),
       "free_disk_GB": round(free_bytes / 1e9, 3)})
if required_bytes > 0.85 * free_bytes:
    raise RuntimeError("The selected profile does not fit safely on the available disk")

metadata = {
    "signature": DATA_SIGNATURE,
    "dataset_version": DATASET_VERSION,
    "pde": asdict(KSCFG),
    "shape_convention": "[time, channel=1, z, y, x]",
    "dtype": "float32",
    "reference_solver": "2/3-dealiased Fourier ETD2",
    "simulator_preflight": SIMULATOR_PREFLIGHT,
    "spatial_preflight": SPATIAL_PREFLIGHT,
}
if METADATA_PATH.exists():
    old_metadata = json.loads(METADATA_PATH.read_text())
    if old_metadata.get("signature") != DATA_SIGNATURE:
        raise RuntimeError(
            f"Dataset signature mismatch in {DATA_ROOT}. Use a new profile/path or remove stale data.")
else:
    atomic_json(METADATA_PATH, metadata)

generation_rows = []
if RUN_DATA_GENERATION:
    for split, count in SPLIT_COUNTS.items():
        for trajectory in range(count):
            generation_rows.append(generate_trajectory(split, trajectory))
    pd.DataFrame(generation_rows).to_csv(DATA_ROOT / "generation_diagnostics.csv", index=False)
else:
    for split, count in SPLIT_COUNTS.items():
        for trajectory in range(count):
            path = trajectory_path(split, trajectory)
            if not path.exists() or not validate_existing_trajectory(path, split):
                raise FileNotFoundError(f"Missing valid generated trajectory: {path}")

print("Dataset ready:", DATA_ROOT)
if generation_rows:
    display(pd.DataFrame(generation_rows).groupby("split").agg(
        trajectories=("trajectory", "count"),
        min_density=("minimum_density", "min"),
        max_density=("maximum_density", "max"),
        max_mass_drift=("mass_relative_drift", "max"),
        max_cfl=("internal_cfl_max", "max")))


## Training-only normalization required by the released reader


In [ ]:
def all_trajectory_paths(split):
    paths = [trajectory_path(split, i) for i in range(SPLIT_COUNTS[split])]
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing trajectory files: " + ", ".join(missing[:3]))
    return paths


STATS_PATH = DATA_ROOT / f"training_stats_{DATA_SIGNATURE}.json"


@torch.no_grad()
def compute_training_statistics(max_state_frames=96, max_carrier_frames=24):
    candidates = []
    for trajectory, path in enumerate(all_trajectory_paths("train")):
        for frame in range(SPLIT_TRANSITIONS["train"] + 1):
            candidates.append((trajectory, frame, path))
    rng = np.random.default_rng(77321)
    chosen = rng.choice(len(candidates), size=min(max_state_frames, len(candidates)),
                        replace=False)
    state_sum = state_square_sum = state_count = 0.0
    delta_sum = delta_square_sum = delta_count = 0.0
    carrier_square_sum = carrier_count = 0.0
    for position, choice in enumerate(tqdm(chosen, desc="training normalization statistics")):
        trajectory, frame, path = candidates[int(choice)]
        array = np.load(path, mmap_mode="r")
        value = np.asarray(array[frame], dtype=np.float64)
        state_sum += value.sum()
        state_square_sum += np.square(value).sum()
        state_count += value.size
        if frame < SPLIT_TRANSITIONS["train"]:
            delta = np.asarray(array[frame + 1], dtype=np.float64) - value
            delta_sum += delta.sum()
            delta_square_sum += np.square(delta).sum()
            delta_count += delta.size
        if position < max_carrier_frames:
            density = torch.from_numpy(value.astype(np.float32)).to(device)
            carrier = SIM.carrier_from_hat(SIM.density_hat(density))
            carrier_square_sum += float(carrier.square().sum())
            carrier_count += carrier.numel()
    state_mean = state_sum / state_count
    state_var = state_square_sum / state_count - state_mean ** 2
    delta_mean = delta_sum / max(delta_count, 1)
    delta_var = delta_square_sum / max(delta_count, 1) - delta_mean ** 2
    return {
        "state_mean": [float(state_mean)],
        "state_std": [float(math.sqrt(max(state_var, 1e-14)))],
        "delta_mean": [float(delta_mean)],
        "delta_std": [float(math.sqrt(max(delta_var, 1e-14)))],
        "carrier_rms": float(math.sqrt(carrier_square_sum / max(carrier_count, 1))),
        "sampled_state_frames": int(len(chosen)),
        "sampled_carrier_frames": int(min(len(chosen), max_carrier_frames)),
        "data_signature": DATA_SIGNATURE,
    }


if STATS_PATH.exists():
    STATS = json.loads(STATS_PATH.read_text())
    if STATS.get("data_signature") != DATA_SIGNATURE:
        raise RuntimeError("Stale normalization statistics")
else:
    STATS = compute_training_statistics()
    atomic_json(STATS_PATH, STATS)


print("These statistics normalize model inputs, outputs, and losses; they are not display-only.")


print(json.dumps(STATS, indent=2))


assert STATS["state_std"][0] > 1e-6 and STATS["delta_std"][0] > 1e-8
